# Deploy a Managed Online Endpoint

Create a Microsoft Entra-authenticated endpoint, deploy the registered foundation model at zero traffic, invoke it directly, and optionally promote traffic.

**Source:** Adapted from [Azure/azureml-examples simple managed deployment](https://github.com/Azure/azureml-examples/blob/37c3572b3ceafdaaa90ee4503c920cfff899df1f/sdk/python/endpoints/online/managed/online-endpoints-simple-deployment.ipynb) and this repository's endpoint patterns, MIT License.

In [ ]:
from pathlib import Path
import json
import os

from azure.ai.ml import MLClient
from azure.ai.ml.constants import ManagedServiceIdentityType
from azure.ai.ml.entities import (
    CodeConfiguration,
    IdentityConfiguration,
    ManagedIdentityConfiguration,
    ManagedOnlineDeployment,
    ManagedOnlineEndpoint,
)
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / ".env.example").is_file() and (candidate / "pipelines").is_dir():
        WORKSHOP_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run this notebook from inside the workshop folder")
load_dotenv(WORKSHOP_ROOT / ".env", override=True)

credential = AzureCliCredential(tenant_id=os.getenv("AZURE_TENANT_ID") or None)
ml_client = MLClient(credential, os.environ["AZURE_SUBSCRIPTION_ID"], os.environ["AZURE_RESOURCE_GROUP"], os.environ["AZUREML_WORKSPACE_NAME"])
ENDPOINT_NAME = os.environ["WORKSHOP_ENDPOINT_NAME"]
DEPLOYMENT_NAME = os.environ["WORKSHOP_DEPLOYMENT_NAME"]
MODEL_NAME = os.environ["WORKSHOP_MODEL_NAME"]
MODEL_VERSION = os.environ["WORKSHOP_MODEL_VERSION"]
ENVIRONMENT_NAME = os.environ["WORKSHOP_ENVIRONMENT_NAME"]
ENVIRONMENT_VERSION = os.environ["WORKSHOP_ENVIRONMENT_VERSION"]
INSTANCE_TYPE = os.environ["AZUREML_ONLINE_INSTANCE_TYPE"]
PUBLIC_ACCESS = os.getenv("AZUREML_ONLINE_ENDPOINT_PUBLIC_NETWORK_ACCESS", "disabled")
IDENTITY_ID = os.getenv("AZUREML_ONLINE_ENDPOINT_IDENTITY_ID", "").strip()
DEPLOY = os.getenv("DEPLOY_FOUNDATION_ENDPOINT", "false").lower() in {"1", "true", "yes"}
PROMOTE = os.getenv("PROMOTE_FOUNDATION_TRAFFIC", "false").lower() in {"1", "true", "yes"}

In [ ]:
from azure.core.exceptions import ResourceNotFoundError

identity = None
if IDENTITY_ID:
    identity = IdentityConfiguration(
        type=ManagedServiceIdentityType.USER_ASSIGNED,
        user_assigned_identities=[ManagedIdentityConfiguration(resource_id=IDENTITY_ID)],
    )

endpoint_definition = ManagedOnlineEndpoint(
    name=ENDPOINT_NAME,
    description="Azure ML workshop foundation endpoint",
    auth_mode="aad_token",
    identity=identity,
    public_network_access=PUBLIC_ACCESS,
    tags={"workshop": "azureml-h2o", "purpose": "foundations"},
)
deployment_definition = ManagedOnlineDeployment(
    name=DEPLOYMENT_NAME,
    endpoint_name=ENDPOINT_NAME,
    model=f"azureml:{MODEL_NAME}:{MODEL_VERSION}",
    environment=f"azureml:{ENVIRONMENT_NAME}:{ENVIRONMENT_VERSION}",
    code_configuration=CodeConfiguration(
        code=str(WORKSHOP_ROOT / "src/foundations/online"),
        scoring_script="score.py",
    ),
    instance_type=INSTANCE_TYPE,
    instance_count=1,
)

if DEPLOY:
    try:
        endpoint = ml_client.online_endpoints.get(ENDPOINT_NAME)
        print(f"Using existing endpoint: {endpoint.name}")
    except ResourceNotFoundError:
        endpoint = ml_client.online_endpoints.begin_create_or_update(endpoint_definition).result()
        print(f"Created endpoint: {endpoint.name}")

    deployment = ml_client.online_deployments.begin_create_or_update(deployment_definition).result()
    if deployment.provisioning_state != "Succeeded":
        raise RuntimeError(f"Deployment state is {deployment.provisioning_state}")

    response = ml_client.online_endpoints.invoke(
        endpoint_name=ENDPOINT_NAME,
        deployment_name=DEPLOYMENT_NAME,
        request_file=str(WORKSHOP_ROOT / "data/requests/foundation-endpoint.json"),
    )
    body = json.loads(response)
    assert body["predictions"] == [9.2, 20.2]
    print(body)

    if PROMOTE:
        endpoint = ml_client.online_endpoints.get(ENDPOINT_NAME)
        endpoint.traffic = {DEPLOYMENT_NAME: 100}
        ml_client.online_endpoints.begin_create_or_update(endpoint).result()
        print(f"Traffic promoted to {DEPLOYMENT_NAME}")
    else:
        print("Traffic remains unchanged. Set PROMOTE_FOUNDATION_TRAFFIC=true to promote.")
else:
    print(f"Prepared endpoint/deployment: {ENDPOINT_NAME}/{DEPLOYMENT_NAME}")
    print("Deployment disabled. Set DEPLOY_FOUNDATION_ENDPOINT=true in workshop/.env.")

## Expected Result

The deployment reaches `Succeeded`, direct invocation returns `[9.2, 20.2]`, and endpoint traffic changes only when the promotion switch is enabled.

Next: `../02_jobs_and_pipelines/01_submit_single_step_merge.ipynb`.